In [14]:
#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.stem import SnowballStemmer
from collections import Counter
import regex  

In [15]:
#ensure tokenizer is available
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\cmetzing\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

In [2]:
#increase print limits 
pd.set_option('display.max_rows', None)

In [28]:
#read data
df = pd.read_csv(r'C:/Users/cmetzing/Downloads/b0cd514b-b9cc-4972-a0c2-c91726e6d825.csv')

In [29]:
#check out data
df.head(1)

,question_id,question_user_id,question_language,question_content,question_topic,question_sent,response_id,response_user_id,response_language,response_content,...,question_user_country_code,question_user_gender,question_user_dob,question_user_created_at,response_user_type,response_user_status,response_user_country_code,response_user_gender,response_user_dob,response_user_created_at
0,3849056,519124,nyn,E ABA WEFARM OFFICES ZABO NIZISHANGWA NKAHI?,NaN,2017-11-22 12:25:03+00,20691011,200868,nyn,E!23 Omubazi Ni Dudu Cipa',...,ug,NaN,NaN,2017-11-18 13:09:11+00,farmer,live,ug,NaN,NaN,2017-05-09 09:19:33+00


In [30]:
df.dtypes

question_id                    int64
question_user_id               int64
question_language             object
question_content              object
question_topic                object
question_sent                 object
response_id                    int64
response_user_id               int64
response_language             object
response_content              object
response_topic                object
response_sent                 object
question_user_type            object
question_user_status          object
question_user_country_code    object
question_user_gender          object
question_user_dob             object
question_user_created_at      object
response_user_type            object
response_user_status          object
response_user_country_code    object
response_user_gender          object
response_user_dob             object
response_user_created_at      object
dtype: object

In [31]:
#possible language values 
df['question_language'].value_counts()

question_language
eng    11976781
swa     6480848
nyn     1157228
lug      689986
Name: count, dtype: int64

In [32]:
#extract eng data
dfe = df[df['question_language'] == 'eng']

In [33]:
#confirm extraction 
dfe['question_language'].value_counts()

question_language
eng    11976781
Name: count, dtype: int64

In [34]:
#check out Eng data
dfe.head(1)

,question_id,question_user_id,question_language,question_content,question_topic,question_sent,response_id,response_user_id,response_language,response_content,...,question_user_country_code,question_user_gender,question_user_dob,question_user_created_at,response_user_type,response_user_status,response_user_country_code,response_user_gender,response_user_dob,response_user_created_at
1,3849061,521327,eng,Q this goes to wefarm. is it possible to get f...,NaN,2017-11-22 12:25:05+00,4334249,526113,eng,Q1 which stage is marleks last vaccinated,...,ug,NaN,NaN,2017-11-20 11:55:48+00,farmer,zombie,ug,NaN,NaN,2017-11-22 10:13:03+00


In [36]:
#reduce dataset to extract question since single question may have multiple responses 
dfe = dfe[["question_id", "question_user_id","question_content","question_topic","question_sent","question_user_country_code"]]

In [37]:
#keep only one row per unique question
dfe = dfe.drop_duplicates(subset="question_id", keep="first")

In [46]:
#inspect
dfe.head()

,question_id,question_user_id,question_content,question_topic,question_sent,question_user_country_code
1,3849061,521327,Q this goes to wefarm. is it possible to get f...,NaN,2017-11-22 12:25:05+00,ug
9,3849084,6642,Q-i have stock rabbit's urine for 5 weeks mash...,rabbit,2017-11-22 12:25:10+00,ke
15,3849098,526375,Q J Have Mi 10000 Can J Start Aproject Of Pout...,poultry,2017-11-22 12:25:12+00,ug
16,3849100,237506,WHERE DO I GET SEEDS OF COCONUT?,pig,2017-11-22 12:25:12+00,ke
23,3849129,54426,Q#.Which plant has omega3?,plant,2017-11-22 12:25:16+00,ke


In [56]:
dfe.describe(include="all")

,question_id,question_user_id,question_content,question_topic,question_sent,question_user_country_code
count,2.940542e+06,2.940542e+06,2940542,2218544,2940542,2940542
unique,NaN,NaN,2708132,148,2939086,4
top,NaN,NaN,What is crop rotation?,chicken,2018-02-01 17:55:53+00,ke
freq,NaN,NaN,839,265257,7,1665002
mean,2.931697e+07,1.659636e+06,NaN,NaN,NaN,NaN
std,1.615944e+07,1.009599e+06,NaN,NaN,NaN,NaN
min,3.849061e+06,7.000000e+00,NaN,NaN,NaN,NaN
25%,1.533666e+07,8.940460e+05,NaN,NaN,NaN,NaN
50%,2.762343e+07,1.428648e+06,NaN,NaN,NaN,NaN
75%,4.305158e+07,2.477375e+06,NaN,NaN,NaN,NaN


In [38]:
#check countries for where question user
dfe['question_user_country_code'].value_counts()

question_user_country_code
ke    1665002
ug    1275419
gb        118
tz          3
Name: count, dtype: int64

In [55]:
#check for nulls
dfe.isnull().sum()  #looks like we are missing question topic a lot

question_id                        0
question_user_id                   0
question_content                   0
question_topic                721998
question_sent                      0
question_user_country_code         0
dtype: int64

In [58]:
#missing
(721998/2940542)*100	

24.55322862247844

In [52]:
#check question topics
dfe['question_topic'].value_counts()

question_topic
chicken               265257
cattle                253285
maize                 200532
tomato                136091
plant                 123615
pig                    88258
bean                   81836
crop                   78338
banana                 70591
poultry                65525
potato                 56107
rabbit                 54606
cabbage                52850
animal                 49112
goat                   48781
coffee                 45644
onion                  33328
watermelon             26331
sheep                  23848
kale                   23210
wheat                  21402
bee                    21112
livestock              21106
passion-fruit          19541
cassava                18538
peanut                 17918
tree                   17811
tea                    17310
rice                   17176
fish                   15218
sugar-cane             14485
carrot                 13845
bird                   12240
grass                  11950

In [60]:
#basic word stem counting setup
stemmer = SnowballStemmer("english")
counter = Counter()

def process_text(text):
    #lowercase and remove non-alphabetic characters
    tokens = regex.findall(r'\b\w+\b', str(text).lower())
    stems = [stemmer.stem(token) for token in tokens]
    return stems

In [61]:
# Process each row and update stem counts
for row in dfe['question_content']:
    stems = process_text(row)
    counter.update(stems)

In [42]:
counter

Counter({'q': 1549075,
         'is': 964418,
         'i': 962786,
         'what': 938189,
         'to': 896895,
         'the': 884224,
         'of': 880628,
         'how': 699114,
         'can': 685831,
         'in': 672002,
         'which': 473894,
         'a': 462172,
         'my': 364652,
         'do': 362454,
         'and': 354005,
         'for': 353014,
         'plant': 339330,
         'are': 272405,
         'it': 252534,
         'best': 244175,
         'maiz': 240688,
         'use': 226725,
         'be': 186732,
         'cow': 186037,
         'have': 166749,
         'type': 157318,
         'get': 155301,
         'that': 153705,
         'tomato': 152130,
         'on': 147548,
         'farm': 141594,
         'ask': 139700,
         'where': 131389,
         'when': 116454,
         'doe': 115863,
         'one': 114972,
         'mani': 114723,
         'by': 114275,
         'from': 112289,
         'grow': 107712,
         'me': 107331,
         'go

In [62]:
#write counts to csv
counts = pd.DataFrame(counter.items(), columns=['item', 'count'])
counts.to_csv('allstemcounter_output.csv', index=False)

In [89]:
#manual inspect key stems
target_stem = "unprofit"
matching_questions = []

for row in dfe['question_content']:
    stems = process_text(row)
    if target_stem in stems:
        matching_questions.append(row)

# Now matching_questions contains all question_contents with the chosen stem
for q in matching_questions:
    print(q)

Q In usin-Gishu maize farming as become unprofitable which other crops can u advise the farmers of uasin-gishu to grow
q which crop is unprofitable in uganda?
s if kept over that age by many weeks they are likely to be unprofitable.the enterprise must be organised to have the contracted number ready for marketi
Q what makes layers poultry unprofitable to keep
Q what makes layers poultry unprofitable to keep
Q A farmer asks: what makes layers poultry unprofitable to keep?
Jina asks: A farmer asks: what makes layers poultry unprofitable to keep?
Q88:Jina asks: A farmer asks: what makes layers poultry unprofitable to keep?


In [99]:
keywords = ["unprofitable"]
keywords_lower = set(k.lower() for k in keywords)

def contains_keyword(text):
    # Tokenize words only (ignore punctuation)
    tokens = set(regex.findall(r'\b\w+\b', str(text).lower()))
    return bool(tokens & keywords_lower)

dfe['contains_keyword'] = dfe['question_content'].apply(contains_keyword)

In [100]:
dfe.head()

,question_id,question_user_id,question_content,question_topic,question_sent,question_user_country_code,contains_keyword,contains_keyworda
1,3849061,521327,Q this goes to wefarm. is it possible to get f...,NaN,2017-11-22 12:25:05+00,ug,False,False
9,3849084,6642,Q-i have stock rabbit's urine for 5 weeks mash...,rabbit,2017-11-22 12:25:10+00,ke,False,False
15,3849098,526375,Q J Have Mi 10000 Can J Start Aproject Of Pout...,poultry,2017-11-22 12:25:12+00,ug,False,False
16,3849100,237506,WHERE DO I GET SEEDS OF COCONUT?,pig,2017-11-22 12:25:12+00,ke,False,False
23,3849129,54426,Q#.Which plant has omega3?,plant,2017-11-22 12:25:16+00,ke,False,False


In [101]:
subset_dfe = dfe[dfe["contains_keyword"] == True]

In [102]:
subset_dfe

,question_id,question_user_id,question_content,question_topic,question_sent,question_user_country_code,contains_keyword,contains_keyworda
2858040,11144259,1077212,Q In usin-Gishu maize farming as become unprof...,crop,2018-09-15 17:49:18.572874+00,ke,True,True
11824279,34554053,2219636,q which crop is unprofitable in uganda?,crop,2019-11-22 16:04:13.135536+00,ug,True,True
15341368,45857192,33010,s if kept over that age by many weeks they are...,NaN,2020-08-13 16:50:06.549966+00,ke,True,True
20292380,59204195,3510445,Q what makes layers poultry unprofitable to keep,chicken,2022-02-03 13:06:43.99644+00,ke,True,True
20292382,59204197,3768170,Q what makes layers poultry unprofitable to keep,chicken,2022-02-03 13:07:27.234141+00,ke,True,True
20292403,59204403,3745919,Q A farmer asks: what makes layers poultry unp...,chicken,2022-02-03 16:12:46.811369+00,ke,True,True
20292414,59204455,3614926,Jina asks: A farmer asks: what makes layers po...,chicken,2022-02-03 16:39:04.854541+00,ke,True,True
20292446,59204457,3614926,Q88:Jina asks: A farmer asks: what makes layer...,chicken,2022-02-03 16:40:42.618912+00,ke,True,True
